# Notebook 12: Trajectory Drift Correction

**R5 — Practical Payoff: Trajectory Correction via 3-Axis Error Subtraction**

This notebook demonstrates the navigational utility of learned drift predictions:
we subtract the predicted 3-axis error vector from each window's dead-reckoned endpoint
and reconstruct continuous flight paths for **MH_01_easy** and **MH_02_easy**.

In [ ]:
import sys
from pathlib import Path
root = Path('.').resolve().parent
sys.path.insert(0, str(root / 'src'))

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

TRAJ_DIR = root / 'results' / 'trajectories'
CORR_CSV = root / 'results' / 'ablations' / 'correction_c1.csv'
FIG_DIR = root / 'figures' / 'trajectories'
print('Trajectories directory:', TRAJ_DIR)


## Section 1: Correction Summary Table (C1)

Comparison of endpoint MAE and cumulative Absolute Trajectory Error (ATE) before and after correction.

In [ ]:
df_corr = pd.read_csv(CORR_CSV)
cols = ['sequence', 'windows_count', 'endpoint_mae_before_cm', 'endpoint_mae_after_cm',
        'endpoint_reduction_pct', 'cumulative_ate_raw_m', 'cumulative_ate_corr_m',
        'final_drift_raw_m', 'final_drift_corr_m', 'ate_reduction_pct']
print(df_corr[cols].to_string(index=False))


## Section 2: MH_01_easy Trajectory Correction

Plotting the reconstructed ground-truth, raw dead-reckoning, and E4vec-corrected paths.

In [ ]:
df1 = pd.read_csv(TRAJ_DIR / 'trajectory_MH_01_easy.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# XY path
ax1.plot(df1['true_x'], df1['true_y'], 'g-', linewidth=2.0, label='Ground truth', zorder=4)
ax1.plot(df1['raw_x'],  df1['raw_y'],  'r--', linewidth=1.2, alpha=0.7, label='Dead-reckoned (raw)', zorder=3)
ax1.plot(df1['corr_x'], df1['corr_y'], 'b-',  linewidth=1.5, label='Corrected (E4vec)', zorder=4)
ax1.scatter([df1['true_x'].iloc[0]], [df1['true_y'].iloc[0]], c='g', s=80, marker='^', label='Start', zorder=5)
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)')
ax1.set_title('MH_01_easy: XY Trajectory Path')
ax1.legend(fontsize=9); ax1.grid(alpha=0.4); ax1.set_aspect('equal')

# ATE vs step
ax2.plot(df1['step'], df1['ate_raw'],  'r-', linewidth=1.5, label='Raw ATE',       alpha=0.8)
ax2.plot(df1['step'], df1['ate_corr'], 'b-', linewidth=1.5, label='Corrected ATE', alpha=0.8)
ax2.set_xlabel('Window step index (2.0 s stride)'); ax2.set_ylabel('ATE (m)')
ax2.set_title('MH_01_easy: ATE vs Step')
ax2.legend(fontsize=9); ax2.grid(alpha=0.4)
plt.suptitle('MH_01_easy Trajectory Correction', fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Final ATE raw={df1['ate_raw'].iloc[-1]:.2f} m -> corrected={df1['ate_corr'].iloc[-1]:.2f} m")


## Section 3: MH_02_easy Trajectory Correction

In [ ]:
df2 = pd.read_csv(TRAJ_DIR / 'trajectory_MH_02_easy.csv')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(df2['true_x'], df2['true_y'], 'g-', linewidth=2.0, label='Ground truth', zorder=4)
ax1.plot(df2['raw_x'],  df2['raw_y'],  'r--', linewidth=1.2, alpha=0.7, label='Dead-reckoned (raw)', zorder=3)
ax1.plot(df2['corr_x'], df2['corr_y'], 'b-',  linewidth=1.5, label='Corrected (E4vec)', zorder=4)
ax1.scatter([df2['true_x'].iloc[0]], [df2['true_y'].iloc[0]], c='g', s=80, marker='^', zorder=5)
ax1.set_xlabel('X (m)'); ax1.set_ylabel('Y (m)')
ax1.set_title('MH_02_easy: XY Trajectory Path')
ax1.legend(fontsize=9); ax1.grid(alpha=0.4); ax1.set_aspect('equal')
ax2.plot(df2['step'], df2['ate_raw'],  'r-', linewidth=1.5, label='Raw ATE',       alpha=0.8)
ax2.plot(df2['step'], df2['ate_corr'], 'b-', linewidth=1.5, label='Corrected ATE', alpha=0.8)
ax2.set_xlabel('Window step index (2.0 s stride)'); ax2.set_ylabel('ATE (m)')
ax2.set_title('MH_02_easy: ATE vs Step')
ax2.legend(fontsize=9); ax2.grid(alpha=0.4)
plt.suptitle('MH_02_easy Trajectory Correction', fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Final ATE raw={df2['ate_raw'].iloc[-1]:.2f} m -> corrected={df2['ate_corr'].iloc[-1]:.2f} m")


## Section 4: Overlay Comparison — Cumulative ATE over Time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(df1['step'], df1['ate_raw'],  color='#E53935', linewidth=1.4, linestyle='--', label='MH_01 Raw Dead-Reckoning')
ax.plot(df1['step'], df1['ate_corr'], color='#1E88E5', linewidth=1.8,                 label='MH_01 Corrected (E4vec)')
ax.plot(df2['step'], df2['ate_raw'],  color='#FB8C00', linewidth=1.4, linestyle='--', label='MH_02 Raw Dead-Reckoning')
ax.plot(df2['step'], df2['ate_corr'], color='#43A047', linewidth=1.8,                 label='MH_02 Corrected (E4vec)')
ax.set_xlabel('Window step index (2.0 s non-overlapping stride)'); ax.set_ylabel('ATE (m)')
ax.set_title('Cumulative Absolute Trajectory Error: Raw Dead-Reckoning vs E4vec Correction')
ax.legend(fontsize=9); ax.grid(alpha=0.4); plt.tight_layout(); plt.show()


## Section 5: Correction Method and Quantitative Conclusion\n\n**Method:**\n1. Select non-overlapping 2.0 s windows (step = 8).\n2. Obtain 3-axis predicted drift vector $[\hat{e}_x, \hat{e}_y, \hat{e}_z]$ from the E4vec multi-task head.\n3. Subtract predicted displacement: $\mathbf{p}_{\text{corrected}} = \mathbf{p}_{\text{dead-reckoned}} - \hat{\mathbf{e}}$.\n4. Step-by-step path chaining for continuous trajectory evaluation.\n\n**Results:**\n- **MH_01_easy:** Endpoint MAE 121.5 cm $\to$ **6.54 cm** (94.6% reduction); Final drift 87.1 m $\to$ **1.49 m** (**98.1% ATE reduction**).\n- **MH_02_easy:** Endpoint MAE 120.0 cm $\to$ **5.37 cm** (95.5% reduction); Final drift 72.1 m $\to$ **0.83 m** (**98.5% ATE reduction**).\n\n**Verification:**\n- Unit test `test_correction.py` proves zero-prediction invariance, perfect-vector zero residual error, and >80% empirical reduction.